# Pre-Training Data Validation — Deep Scan

Augments `test_preencoded_dataset.ipynb` (which samples 5 shards) with a systematic scan
over a configurable fraction of all shards. **Runs a single streaming pass** to compute:

- Systematic NaN/Inf detection with exact shard+row location reporting
- Token value statistics and dead-dimension detection (V-JEPA2 encoder health)
- Per-dimension action and state statistics with range/dead-channel flags
- Task distribution from manifest (episodes and hours per task)
- Multi-camera completeness (wrist views non-zero)

**Runtime target:** < 15 min at `SCAN_FRACTION=0.20` on local NVMe.

In [ ]:
import collections
import collections.abc
for _name in ("Iterator", "Mapping", "MutableMapping", "Sequence", "MutableSequence", "Set"):
    if not hasattr(collections, _name):
        setattr(collections, _name, getattr(collections.abc, _name))

import json, os, tempfile, warnings
from collections import defaultdict
from math import ceil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams["figure.dpi"] = 110
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from streaming import StreamingDataset

# ── User config ───────────────────────────────────────────────────────────────
LOCAL_SHARD_DIR = Path("/workspace/shards_256px_vit_16_g")
MANIFEST_PATH   = Path("/workspace/base_dataset/manifest.json")
VIEWS           = ["head", "left_wrist", "right_wrist"]

SCAN_FRACTION   = 0.20   # fraction of shards to scan
SEED            = 42

# ── Dataset constants (must match encoding config) ────────────────────────────
TARGET_FPS      = 5
ACTION_DIM      = 23
STATE_DIM       = 133
NATIVE_FPS      = 30
FSTP_NOM        = ceil(NATIVE_FPS / TARGET_FPS)    # = 6
ACTIONS_PER_ROW = FSTP_NOM * ACTION_DIM             # = 138

# ── Validation thresholds ─────────────────────────────────────────────────────
TOKEN_DEAD_STD_THRESHOLD          = 1e-4
TOKEN_MEAN_WARNING                = 2.0
TOKEN_STD_WARNING_LOW             = 0.3
TOKEN_STD_WARNING_HIGH            = 3.0
ACTION_STATE_DEAD_STD             = 1e-3
ACTION_STATE_RANGE_MAX            = 100.0

# ── Proprio slices (must match behavior.py PROPRIO_SLICES) ───────────────────
PROPRIO_SLICES = [
    np.s_[6:28], np.s_[34:56], np.s_[62:84], np.s_[84:112],
    np.s_[152:158], np.s_[186:197], np.s_[225:244], np.s_[253:256],
]

def extract_proprio(full_states: np.ndarray) -> np.ndarray:
    return np.concatenate([full_states[..., s] for s in PROPRIO_SLICES], axis=-1)

print(f"LOCAL_SHARD_DIR : {LOCAL_SHARD_DIR}")
print(f"MANIFEST_PATH   : {MANIFEST_PATH}")
print(f"SCAN_FRACTION   : {SCAN_FRACTION}  SEED={SEED}")
print(f"VIEWS           : {VIEWS}")
print(f"ACTIONS_PER_ROW : {ACTIONS_PER_ROW}  STATE_DIM={STATE_DIM}")

In [ ]:
class WelfordAccumulator:
    """Online mean/variance (Welford) + reservoir for p1/p99."""

    def __init__(self, shape):
        shape = (shape,) if isinstance(shape, int) else tuple(shape)
        self.n    = 0
        self.mean = np.zeros(shape, dtype=np.float64)
        self.M2   = np.zeros(shape, dtype=np.float64)
        self.min_ = np.full(shape,  np.inf,  dtype=np.float64)
        self.max_ = np.full(shape, -np.inf, dtype=np.float64)
        self._reservoir_size  = 50_000
        self._reservoir       = []
        self._reservoir_count = 0

    def update(self, x: np.ndarray):
        x = np.asarray(x, dtype=np.float64)
        self.n   += 1
        delta     = x - self.mean
        self.mean += delta / self.n
        self.M2   += delta * (x - self.mean)
        self.min_  = np.minimum(self.min_, x)
        self.max_  = np.maximum(self.max_, x)

    def update_reservoir(self, flat_values: np.ndarray):
        rng = np.random.default_rng()  # thread-local, fine for notebook
        for v in flat_values.ravel():
            self._reservoir_count += 1
            if len(self._reservoir) < self._reservoir_size:
                self._reservoir.append(float(v))
            else:
                j = int(rng.integers(0, self._reservoir_count))
                if j < self._reservoir_size:
                    self._reservoir[j] = float(v)

    @property
    def std(self):
        if self.n < 2:
            return np.zeros_like(self.mean)
        return np.sqrt(np.maximum(self.M2 / (self.n - 1), 0.0))

    def percentiles(self, q):
        if not self._reservoir:
            return np.nan
        return float(np.percentile(self._reservoir, q))

print("WelfordAccumulator defined.")

---
## Step 1 — Index Validation & Shard Selection (instant)

In [ ]:
assert LOCAL_SHARD_DIR.exists(), f"Shard dir not found: {LOCAL_SHARD_DIR}"
index_path = LOCAL_SHARD_DIR / "index.json"
assert index_path.exists(), "index.json missing"

with open(index_path) as f:
    index = json.load(f)

shards     = index.get("shards", [])
assert len(shards) > 0, "No shards in index.json"

total_rows  = sum(s["samples"] for s in shards)
total_bytes = sum(s.get("raw_data", {}).get("size", 0) for s in shards)

TOKEN_COLS    = {f"tokens_{v}" for v in VIEWS}
EXPECTED_COLS = TOKEN_COLS | {
    "actions", "states", "cam_rel_poses",
    "frame_index", "episode_idx", "sample_idx", "step_pos", "episode_len"
}
schema_issues = []
for s in shards:
    missing = EXPECTED_COLS - set(s.get("column_names", []))
    if missing:
        schema_issues.append(f"  {s['raw_data']['basename']}: missing {missing}")
if schema_issues:
    for msg in schema_issues[:5]: print("SCHEMA FAIL:", msg)
    raise AssertionError(f"{len(schema_issues)} shards have schema issues")

print(f"Total shards : {len(shards):,}")
print(f"Total rows   : {total_rows:,}")
print(f"Total size   : {total_bytes / 1e12:.3f} TB")
print("Schema OK — all shards have expected columns.")

# Seeded random sample
rng          = np.random.default_rng(SEED)
n_scan       = max(1, int(ceil(SCAN_FRACTION * len(shards))))
shard_idxs   = sorted(rng.choice(len(shards), size=n_scan, replace=False).tolist())
sampled_shards = [shards[i] for i in shard_idxs]
sampled_rows   = sum(s["samples"] for s in sampled_shards)

print(f"\nScanning {n_scan}/{len(shards)} shards ({100*SCAN_FRACTION:.0f}%) → {sampled_rows:,} rows")

In [ ]:
tmp_dir = tempfile.mkdtemp(prefix="behavior_deepval_")
for si in sampled_shards:
    basename = si["raw_data"]["basename"]
    src = LOCAL_SHARD_DIR / basename
    dst = Path(tmp_dir) / basename
    assert src.exists(), f"Shard file missing: {src}"
    if not dst.exists():
        os.symlink(src, dst)

sub_index = {"version": index.get("version", 2), "shards": sampled_shards}
with open(Path(tmp_dir) / "index.json", "w") as f:
    json.dump(sub_index, f)

scan_ds   = StreamingDataset(local=tmp_dir, shuffle=False)
n_scan_ds = len(scan_ds)
print(f"StreamingDataset ready: {n_scan_ds:,} rows in {len(sampled_shards)} shards")
print(f"Temp dir: {tmp_dir}")

---
## Step 2 — Single-Pass Streaming Scan

Iterates every row in the sampled shards **exactly once**.
All statistics are accumulated with Welford online accumulators — no buffering of arrays.

Tracks: NaN/Inf locations · token per-dim mean/std · wrist zero rows · action/state range · shape failures.

In [ ]:
# ── Accumulators (defined here so loop cell can be re-run after interruption) ─
TOKEN_SHAPE = None
EMBED_DIM   = None

tok_acc        = {}   # view → WelfordAccumulator(embed_dim,)  per-dim
tok_global_acc = {}   # view → WelfordAccumulator(())          reservoir only
act_acc        = WelfordAccumulator(shape=(ACTIONS_PER_ROW,))
state_acc      = WelfordAccumulator(shape=(STATE_DIM,))

nan_inf_locs      = []   # (global_row_idx, shard_basename, field)
wrist_zero_counts = {v: 0 for v in ["left_wrist", "right_wrist"]}
shape_failures    = []
range_flags       = []   # (global_row_idx, field, dim_list, val_list)

# Shard boundary lookup for location reporting
_cumulative = []
_running = 0
for _si in sampled_shards:
    _running += _si["samples"]
    _cumulative.append((_running, _si["raw_data"]["basename"]))

def _shard_for_row(ridx):
    for boundary, name in _cumulative:
        if ridx < boundary:
            return name
    return _cumulative[-1][1]

print("Accumulators initialized — ready for scan loop.")

In [ ]:
for global_row_idx in tqdm(range(n_scan_ds), desc="Scanning", unit="row", smoothing=0.02):
    row        = scan_ds[global_row_idx]
    shard_name = _shard_for_row(global_row_idx)

    # ── First-row shape detection ────────────────────────────────────────────
    if TOKEN_SHAPE is None:
        tok0 = row[f"tokens_{VIEWS[0]}"]
        assert tok0.ndim == 2, f"Expected 2D token array, got shape {tok0.shape}"
        TOKEN_SHAPE = tok0.shape
        EMBED_DIM   = TOKEN_SHAPE[1]
        for v in VIEWS:
            tok_acc[v]        = WelfordAccumulator(shape=(EMBED_DIM,))
            tok_global_acc[v] = WelfordAccumulator(shape=(1,))
        tqdm.write(f"  Detected TOKEN_SHAPE={TOKEN_SHAPE}  EMBED_DIM={EMBED_DIM}")

    # ── Token checks & accumulation ──────────────────────────────────────────
    for v in VIEWS:
        tok = row[f"tokens_{v}"].astype(np.float32)

        if tok.shape != TOKEN_SHAPE:
            shape_failures.append(
                f"row {global_row_idx} ({shard_name}) {v}: {tok.shape} != {TOKEN_SHAPE}"
            )

        if not np.all(np.isfinite(tok)):
            nan_inf_locs.append((global_row_idx, shard_name, f"tokens_{v}"))

        if v in ("left_wrist", "right_wrist") and np.all(tok == 0.0):
            wrist_zero_counts[v] += 1

        # Welford: average across tokens_per_frame → (embed_dim,) per-dim mean
        tok_acc[v].update(tok.mean(axis=0).astype(np.float64))

        # Reservoir: subsample every 50 rows to keep cost low
        if global_row_idx % 50 == 0:
            tok_global_acc[v].update_reservoir(tok.ravel())

    # ── Action checks & accumulation ─────────────────────────────────────────
    act = row["actions"].astype(np.float32)
    if act.shape != (ACTIONS_PER_ROW,):
        shape_failures.append(
            f"row {global_row_idx}: actions shape {act.shape} != ({ACTIONS_PER_ROW},)"
        )
    if not np.all(np.isfinite(act)):
        nan_inf_locs.append((global_row_idx, shard_name, "actions"))
    bad = np.where(np.abs(act) > ACTION_STATE_RANGE_MAX)[0]
    if len(bad) > 0:
        range_flags.append((global_row_idx, "actions", bad.tolist(), act[bad].tolist()))
    act_acc.update(act.astype(np.float64))

    # ── State checks & accumulation ───────────────────────────────────────────
    st = row["states"].astype(np.float32)
    if st.shape != (STATE_DIM,):
        shape_failures.append(
            f"row {global_row_idx}: states shape {st.shape} != ({STATE_DIM},)"
        )
    if not np.all(np.isfinite(st)):
        nan_inf_locs.append((global_row_idx, shard_name, "states"))
    bad = np.where(np.abs(st) > ACTION_STATE_RANGE_MAX)[0]
    if len(bad) > 0:
        range_flags.append((global_row_idx, "states", bad.tolist(), st[bad].tolist()))
    state_acc.update(st.astype(np.float64))

print(f"\nScan complete. {n_scan_ds:,} rows processed.")
print(f"NaN/Inf locations  : {len(nan_inf_locs)}")
print(f"Shape failures     : {len(shape_failures)}")
print(f"Range flags        : {len(range_flags)}")

---
## Check 1 — Systematic NaN/Inf Report

In [ ]:
nan_check_pass = len(nan_inf_locs) == 0

if nan_check_pass:
    print(f"PASS — No NaN/Inf found across {n_scan_ds:,} scanned rows.")
else:
    warnings.warn(
        f"NaN/Inf detected in {len(nan_inf_locs)} row-field combinations "
        f"out of {n_scan_ds:,} rows scanned."
    )
    df_nans = pd.DataFrame(nan_inf_locs, columns=["global_row_idx", "shard_basename", "field"])
    print(f"\nLocations (first 50):")
    print(df_nans.head(50).to_string(index=False))
    print("\nBy field:")
    print(df_nans.groupby("field").size().to_string())
    print("\nBy shard (top 20):")
    print(df_nans.groupby("shard_basename").size().sort_values(ascending=False).head(20).to_string())

---
## Check 2 — Token Value Statistics & Histograms

In [ ]:
token_check_pass = True
n_views = len(VIEWS)
fig, axes = plt.subplots(1, n_views, figsize=(5 * n_views, 4), sharey=False)
if n_views == 1:
    axes = [axes]

colors = ["steelblue", "darkorange", "green"]
for ax, v, color in zip(axes, VIEWS, colors):
    acc  = tok_acc[v]
    gacc = tok_global_acc[v]

    global_mean      = float(acc.mean.mean())
    mean_perdim_std  = float(acc.std.mean())
    p1               = gacc.percentiles(1)
    p99              = gacc.percentiles(99)
    gmin             = float(acc.min_.min())
    gmax             = float(acc.max_.max())

    print(f"\n--- tokens_{v} ---")
    print(f"  global mean (mean of per-dim means) : {global_mean:+.4f}")
    print(f"  mean per-dim std (Welford)          : {mean_perdim_std:.4f}")
    print(f"  global min / max                    : {gmin:.4f} / {gmax:.4f}")
    print(f"  p1 / p99 (reservoir)                : {p1:.4f} / {p99:.4f}")

    if abs(global_mean) > TOKEN_MEAN_WARNING:
        warnings.warn(f"tokens_{v}: |global mean| = {abs(global_mean):.4f} > {TOKEN_MEAN_WARNING}")
        token_check_pass = False
    if mean_perdim_std < TOKEN_STD_WARNING_LOW or mean_perdim_std > TOKEN_STD_WARNING_HIGH:
        warnings.warn(
            f"tokens_{v}: mean per-dim std = {mean_perdim_std:.4f} "
            f"outside [{TOKEN_STD_WARNING_LOW}, {TOKEN_STD_WARNING_HIGH}]"
        )
        token_check_pass = False

    ax.hist(acc.mean, bins=80, color=color, alpha=0.8, edgecolor="none")
    ax.axvline(0, color="black", linestyle="--", linewidth=0.8, label="zero")
    ax.set_title(f"tokens_{v}\nper-dim mean distribution")
    ax.set_xlabel("Welford mean per embedding dim")
    ax.set_ylabel("# dims")
    ax.legend(fontsize=8)

plt.suptitle("Token Per-Dim Mean Distribution (V-JEPA2 output should be approx. centred)", y=1.02)
plt.tight_layout()
plt.show()

print("\nPASS" if token_check_pass else "\nWARNING — token stats outside expected bounds; see above.")

---
## Check 3 — Dead Token Dimension Detection

In [ ]:
dead_check_pass = True

for v in VIEWS:
    acc       = tok_acc[v]
    dead_dims = np.where(acc.std < TOKEN_DEAD_STD_THRESHOLD)[0]
    pct       = 100.0 * len(dead_dims) / EMBED_DIM
    print(f"tokens_{v}: {len(dead_dims)}/{EMBED_DIM} dead dims (std < {TOKEN_DEAD_STD_THRESHOLD})  [{pct:.2f}%]")

    if len(dead_dims) > 0:
        warnings.warn(
            f"tokens_{v}: {len(dead_dims)} dead embedding dims. First 20: {dead_dims[:20].tolist()}"
        )
        dead_check_pass = False
        fig, ax = plt.subplots(figsize=(12, 3))
        ax.bar(range(EMBED_DIM), acc.std, width=1.0, color="steelblue", alpha=0.7)
        ax.axhline(TOKEN_DEAD_STD_THRESHOLD, color="red", linestyle="--", linewidth=1.0,
                   label=f"Dead threshold ({TOKEN_DEAD_STD_THRESHOLD})")
        ax.set_xlabel("Embedding dim index")
        ax.set_ylabel("Welford std")
        ax.set_title(f"tokens_{v} — Per-dim Welford Std")
        ax.legend()
        plt.tight_layout()
        plt.show()

print("\nPASS — No dead token dims." if dead_check_pass else "\nWARNING — Dead token dims detected.")

---
## Check 4 — Action Per-Dimension Statistics

In [ ]:
act_mean = act_acc.mean
act_std  = act_acc.std
act_min  = act_acc.min_
act_max  = act_acc.max_

dead_act  = np.where(act_std < ACTION_STATE_DEAD_STD)[0]
range_act = np.where(np.maximum(np.abs(act_min), np.abs(act_max)) > ACTION_STATE_RANGE_MAX)[0]

df_act = pd.DataFrame({"dim": range(ACTIONS_PER_ROW),
                        "mean": act_mean, "std": act_std,
                        "min": act_min, "max": act_max})
print(df_act.to_string())

act_check_pass = True
if len(dead_act) > 0:
    warnings.warn(f"Actions: {len(dead_act)} near-constant dims (std < {ACTION_STATE_DEAD_STD}): {dead_act.tolist()}")
    act_check_pass = False
if len(range_act) > 0:
    warnings.warn(f"Actions: {len(range_act)} out-of-range dims (|value| > {ACTION_STATE_RANGE_MAX}): {range_act.tolist()}")
    act_check_pass = False

fig, axes = plt.subplots(2, 1, figsize=(14, 6))
axes[0].bar(range(ACTIONS_PER_ROW), act_std, width=1.0, color="darkorange", alpha=0.8)
axes[0].axhline(ACTION_STATE_DEAD_STD, color="red", linestyle="--", linewidth=1.0,
                label=f"Dead threshold ({ACTION_STATE_DEAD_STD})")
axes[0].set_title("Action Per-Dim Std")
axes[0].set_xlabel("Action dim index"); axes[0].set_ylabel("Std"); axes[0].legend()

axes[1].bar(range(ACTIONS_PER_ROW), act_mean, width=1.0, color="steelblue", alpha=0.8)
axes[1].set_title("Action Per-Dim Mean")
axes[1].set_xlabel("Action dim index"); axes[1].set_ylabel("Mean")
plt.tight_layout(); plt.show()

print("PASS" if act_check_pass else "WARNING — action dimension issues; see above.")

---
## Check 5 — State Per-Dimension Statistics

In [ ]:
state_mean = state_acc.mean
state_std  = state_acc.std
state_min  = state_acc.min_
state_max  = state_acc.max_

dead_state  = np.where(state_std < ACTION_STATE_DEAD_STD)[0]
range_state = np.where(np.maximum(np.abs(state_min), np.abs(state_max)) > ACTION_STATE_RANGE_MAX)[0]

# Semantic slice boundary labels
SLICE_LABELS = [
    (0,  22,  "joint_qpos"),
    (22, 44,  "joint_qpos_sin"),
    (44, 66,  "joint_qpos_cos"),
    (66, 94,  "joint_qvel"),
    (94, 100, "robot_vel"),
    (100, 111, "eef_left"),
    (111, 130, "eef_right+trunk"),
    (130, 133, "base_qvel"),
]

df_state = pd.DataFrame({"dim": range(STATE_DIM),
                          "mean": state_mean, "std": state_std,
                          "min": state_min, "max": state_max})
print(df_state.to_string())

state_check_pass = True
if len(dead_state) > 0:
    warnings.warn(f"States: {len(dead_state)} near-constant dims: {dead_state.tolist()}")
    state_check_pass = False
if len(range_state) > 0:
    warnings.warn(f"States: {len(range_state)} out-of-range dims: {range_state.tolist()}")
    state_check_pass = False

fig, axes = plt.subplots(2, 1, figsize=(14, 6))
axes[0].bar(range(STATE_DIM), state_std, width=1.0, color="green", alpha=0.8)
axes[0].axhline(ACTION_STATE_DEAD_STD, color="red", linestyle="--", linewidth=1.0,
                label=f"Dead threshold ({ACTION_STATE_DEAD_STD})")
# Slice boundary lines
for start, end, label in SLICE_LABELS:
    axes[0].axvline(start, color="gray", linestyle=":", linewidth=0.7)
    axes[0].text(start + 0.5, axes[0].get_ylim()[1] * 0.9, label,
                 fontsize=6, rotation=45, va="top")
axes[0].set_title("State Per-Dim Std"); axes[0].set_xlabel("State dim"); axes[0].legend()

axes[1].bar(range(STATE_DIM), state_mean, width=1.0, color="purple", alpha=0.6)
for start, _, _ in SLICE_LABELS:
    axes[1].axvline(start, color="gray", linestyle=":", linewidth=0.7)
axes[1].set_title("State Per-Dim Mean"); axes[1].set_xlabel("State dim"); axes[1].set_ylabel("Mean")
plt.tight_layout(); plt.show()

print("PASS" if state_check_pass else "WARNING — state dimension issues; see above.")

---
## Check 6 — Task Distribution from Manifest

In [ ]:
assert MANIFEST_PATH.exists(), f"manifest.json not found: {MANIFEST_PATH}"
with open(MANIFEST_PATH) as f:
    manifest = json.load(f)

episodes_list = manifest.get("episodes", [])
print(f"Manifest episodes: {len(episodes_list)}")

task_ep_count = defaultdict(int)
task_hours    = defaultdict(float)

for ep in episodes_list:
    task = ep.get("task_name") or ep.get("task") or ep.get("task_id") or "unknown"
    task_ep_count[task] += 1
    dur_min = 0.0
    if "duration_minutes" in ep:
        dur_min = float(ep["duration_minutes"])
    elif "duration_seconds" in ep:
        dur_min = float(ep["duration_seconds"]) / 60.0
    elif "duration_hours" in ep:
        dur_min = float(ep["duration_hours"]) * 60.0
    task_hours[task] += dur_min / 60.0

df_tasks = pd.DataFrame({
    "task":       list(task_ep_count.keys()),
    "n_episodes": [task_ep_count[t] for t in task_ep_count],
    "hours":      [task_hours[t]    for t in task_ep_count],
}).sort_values("n_episodes", ascending=False).reset_index(drop=True)

print(f"\nUnique tasks : {len(df_tasks)}")
print(f"Total eps    : {df_tasks['n_episodes'].sum()}")
print(f"Total hours  : {df_tasks['hours'].sum():.2f} h")
print(df_tasks.to_string())

sparse_tasks   = df_tasks[df_tasks["n_episodes"] < 2]["task"].tolist()
task_check_pass = True
if sparse_tasks:
    warnings.warn(f"{len(sparse_tasks)} tasks have < 2 episodes: {sparse_tasks}")
    task_check_pass = False

N_SHOW  = min(40, len(df_tasks))
df_plot = df_tasks.head(N_SHOW)
labels  = [t[:32] + "…" if len(t) > 32 else t for t in df_plot["task"]]

fig, axes = plt.subplots(1, 2, figsize=(18, max(6, N_SHOW * 0.24)))
axes[0].barh(range(N_SHOW), df_plot["n_episodes"], color="steelblue", alpha=0.8)
axes[0].set_yticks(range(N_SHOW)); axes[0].set_yticklabels(labels, fontsize=7)
axes[0].invert_yaxis()
axes[0].axvline(2, color="red", linestyle="--", linewidth=0.8, label="< 2 threshold")
axes[0].set_xlabel("Episodes"); axes[0].set_title(f"Episodes per task (top {N_SHOW})")
axes[0].legend(fontsize=8)

axes[1].barh(range(N_SHOW), df_plot["hours"], color="darkorange", alpha=0.8)
axes[1].set_yticks(range(N_SHOW)); axes[1].set_yticklabels(labels, fontsize=7)
axes[1].invert_yaxis()
axes[1].set_xlabel("Hours"); axes[1].set_title(f"Hours per task (top {N_SHOW})")
plt.tight_layout(); plt.show()

print("PASS" if task_check_pass else f"WARNING — {len(sparse_tasks)} sparse tasks.")

---
## Check 7 — Multi-Camera Completeness

In [ ]:
wrist_check_pass = True
print(f"Rows scanned: {n_scan_ds:,}")
for v in ["left_wrist", "right_wrist"]:
    count = wrist_zero_counts[v]
    pct   = 100.0 * count / n_scan_ds if n_scan_ds > 0 else 0.0
    print(f"  tokens_{v} all-zero rows: {count:,} / {n_scan_ds:,}  ({pct:.3f}%)")
    if count > 0:
        warnings.warn(
            f"tokens_{v}: {count} rows ({pct:.3f}%) have all-zero token arrays. "
            "Check whether multi-view encoding wrote correct camera data."
        )
        wrist_check_pass = False

print("PASS — All wrist rows non-zero." if wrist_check_pass else "WARNING — Some wrist rows are all-zero.")

---
## Check 8 — Shape Consistency & Range Violations

In [ ]:
shape_check_pass = len(shape_failures) == 0
range_check_pass = len(range_flags)    == 0

if shape_check_pass:
    print("PASS — No shape inconsistencies.")
else:
    warnings.warn(f"{len(shape_failures)} shape inconsistencies found.")
    for msg in shape_failures[:20]:
        print("  SHAPE:", msg)

if range_check_pass:
    print("PASS — No action/state values exceed |100|.")
else:
    warnings.warn(f"{len(range_flags)} row-field combinations have |value| > {ACTION_STATE_RANGE_MAX}.")
    for rf in range_flags[:20]:
        ridx, field, dims, vals = rf
        print(f"  RANGE: row {ridx}, {field}, dims={dims[:5]}, values={[round(v,2) for v in vals[:5]]}")

---
## Final Summary

In [ ]:
checks = [
    ("1. Schema",               True,              "All shards have required MDS columns"),
    ("2. NaN/Inf (systematic)", nan_check_pass,    f"{len(nan_inf_locs)} occurrences in {SCAN_FRACTION*100:.0f}% scan"),
    ("3. Token value stats",    token_check_pass,  f"Global mean/std within V-JEPA2 expected bounds"),
    ("4. Dead token dims",      dead_check_pass,   f"std < {TOKEN_DEAD_STD_THRESHOLD} threshold"),
    ("5. Action per-dim",       act_check_pass,    "Near-constant or out-of-range dims"),
    ("6. State per-dim",        state_check_pass,  "Near-constant or out-of-range dims"),
    ("7. Task distribution",    task_check_pass,   f"{len(sparse_tasks)} tasks with < 2 episodes"),
    ("8. Wrist completeness",   wrist_check_pass,  "All wrist token rows non-zero"),
    ("9. Shape consistency",    shape_check_pass,  f"{len(shape_failures)} shape mismatches"),
]

print("=" * 70)
print("VALIDATION SUMMARY")
print("=" * 70)
for name, passed, notes in checks:
    sym = "PASS" if passed else "WARN"
    print(f"{sym:4s}  {name:<35s}  {notes}")
print("=" * 70)

df_summary = pd.DataFrame(checks, columns=["Check", "Passed", "Notes"])
df_summary["Result"] = df_summary["Passed"].map({True: "PASS", False: "WARN"})

def _color_result(val):
    return ("background-color: #d4edda" if val == "PASS"
            else "background-color: #fff3cd")

display(df_summary[["Check", "Result", "Notes"]].style.applymap(_color_result, subset=["Result"]))

# Critical failures block training; advisory failures are informational
critical_pass = nan_check_pass and shape_check_pass
if critical_pass:
    print("\nAll critical checks PASSED. Dataset is ready for pre-training.")
    print("Review any WARNINGs above before launching GPU jobs.")
else:
    raise AssertionError(
        "Critical validation failures detected — address NaN/Inf or shape issues before training."
    )